# 🏆 PLAN 1: PERFORMANCE — Chế Độ Chuẩn Tesla T4 (16GB VRAM • ~1.53 CU/h)

> **Khi nào sử dụng:** Dùng từ **Đầu tháng $ightarrow$ Giữa/Cuối tháng khi Compute Units còn dư dả ($> 35	ext{ CU}$)** cho các buổi làm việc hàng ngày. Đạt chất lượng code, refactor đa file và thiết kế kiến trúc cao nhất chỉ với **~1.53 CU/h** (code 3–4h/ngày suốt cả tháng).
> *(Tự động thích ứng: Nếu bạn đổi Kernel sang **A100 40GB**, notebook này cũng tự động mở khóa chế độ Dual-Resident `OLLAMA_MAX_LOADED_MODELS=2`!)*

---

## 🧩 Kiến Trúc Nhạc Trưởng & Chuyên Gia — Plan 1 (Tesla T4 16GB)

| Scope công việc | Vai trò trong Protocol | Mô hình trên Tesla T4 (16GB VRAM) | Cơ chế hoạt động & VRAM |
| :--- | :--- | :--- | :--- |
| **1. Planning & Reasoning** | 🎼 **Model 1 — Nhạc Trưởng**<br>*(Chạy đúng 1 lần đầu feature để xuất checklist `T1..Tn` vào `HANDOFF.md`)* | **`deepseek-r1:14b`** | ~9.0 GB VRAM • Suy luận chuỗi tư duy `<think>` sâu sắc, bóc tách kiến trúc & edge-cases. |
| **2. Coding (Trọng tâm 80%)** | 🛠️ **Model 2 — Chuyên Gia**<br>*(Thực thi tuần tự `T1 -> Tn`, viết code & unit test)* | **`qwen2.5-coder:14b` ⭐** | ~9.5 GB VRAM • Dense 14.7B thuần code (~35 tok/s, HumanEval 86.4%). |
| **3. Workflow (CLI / JSON)** | 🛠️ **Model 2 — Chuyên Gia** *(Giữ nguyên Model 2 — 0 lần switch)* | **`qwen2.5-coder:14b`** | Giữ nguyên trong VRAM suốt chuỗi task `T1 -> Tn`. |
| **4. Diagram (Mermaid.js)** | 🛠️ **Model 2 — Chuyên Gia** *(Giữ nguyên Model 2 — 0 lần switch)* | **`qwen2.5-coder:14b`** | Vẽ sơ đồ kiến trúc, Flowchart, Sequence Diagram chuẩn xác. |

---

## 🛡️ Cấu Hình VRAM & Chống Mất Context (82% VRAM — 0% OOM)
- **`OLLAMA_MAX_LOADED_MODELS="1"`** (trên T4 16GB) kết hợp **`OLLAMA_FLASH_ATTENTION="1"`** + **`OLLAMA_KV_CACHE_TYPE="q8_0"`**.
- **Đỉnh VRAM (16K context):** `13.0 GB / 15.84 GB (82% VRAM)` — an toàn tuyệt đối, không bao giờ OOM.
- **Switch Latency (~1.8 giây):** Nhờ **16GB NVMe Swapfile** + **Warmup Linux OS Page Cache**, bạn chỉ chuyển đúng **1 lần duy nhất** từ `deepseek-r1:14b` (M1) sang `qwen2.5-coder:14b` (M2) sau bước lập kế hoạch; toàn bộ quá trình code `T1 -> Tn` tái sử dụng 100% KV Cache Prefix (`< 0.15s` prefill)!

---

💡 **Các Notebook Khác Trong Bộ:**
- 🚀 **Cần sức mạnh tối đa 32B Dual-Resident trên A100 (40GB):** Mở [`colab_ollama_plan1_performance_a100.ipynb`](colab_ollama_plan1_performance_a100.ipynb)
- ⚡ **Cuối tháng khi CU gần cạn ($< 35	ext{ CU}$) hoặc cần gõ tốc độ 75 tok/s:** Mở [`colab_ollama_plan2_fast_t4.ipynb`](colab_ollama_plan2_fast_t4.ipynb)
- 📘 **Bản Tutorial cơ bản (1-Model):** Mở [`colab_ollama_tutorial.ipynb`](colab_ollama_tutorial.ipynb)

### Step 1: Xác Thực Phần Cứng NVIDIA GPU & VRAM
Kiểm tra card GPU (Tesla T4 16GB hoặc A100 40GB) đã được gắn vào máy ảo Google Colab.

In [ ]:
# Step 1: Kiểm tra phần cứng NVIDIA GPU và dung lượng VRAM thực tế
!nvidia-smi

### Step 2: Cấu Hình Lưu Trữ Mô Hình (Colab NVMe SSD Siêu Tốc hoặc Google Drive)
> 💡 **Khuyến nghị:** Mặc định `USE_GDRIVE = False` để sử dụng ổ cứng **Colab NVMe SSD nội bộ (`/root/.ollama/models`)** với tốc độ đọc/ghi >600 MB/s và tránh popup xác thực khi chạy trực tiếp trong **Antigravity IDE Colab Kernel**. Nếu chạy trên trình duyệt web và muốn lưu cache qua ngày, đổi `USE_GDRIVE = True`.

In [ ]:
import os

# Đặt True nếu chạy trên trình duyệt (Safari/Chrome) và muốn lưu model cố định vào Google Drive
# Mặc định False để ưu tiên tốc độ NVMe SSD nội bộ (>600 MB/s) & tương thích 100% với IDE Kernel
USE_GDRIVE = False

if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        models_dir = '/content/drive/MyDrive/ollama_models'
        os.environ['OLLAMA_MODELS'] = models_dir
        os.makedirs(models_dir, exist_ok=True)
        print(f"✅ Đã kích hoạt Google Drive cache tại: {models_dir}")
    except Exception as e:
        print(f"ℹ️ Đang chạy trong IDE Kernel (Bỏ qua Google Drive mount: {e}).")
        print("⚡ Sử dụng ổ cứng Colab NVMe SSD nội bộ (/root/.ollama/models).")
else:
    print("⚡ Sử dụng ổ cứng Colab NVMe SSD nội bộ (/root/.ollama/models) — Tối ưu tốc độ nạp model.")

### Step 3: Cài Đặt Ollama & Khởi Chạy Daemon (Tự Động Thích Ứng T4 16GB / A100 40GB)
- Kích hoạt **16GB Linux NVMe Swapfile** (nâng tổng bộ nhớ ảo Colab T4 từ 12.67GB lên **28.67GB**).
- Bật **`OLLAMA_FLASH_ATTENTION="1"`** và **`OLLAMA_KV_CACHE_TYPE="q8_0"`**.
- Tự động kiểm tra dung lượng VRAM thực tế: đặt `OLLAMA_MAX_LOADED_MODELS="1"` trên T4 16GB (hoặc `"2"` nếu bạn chọn kernel A100 40GB).

In [ ]:
import os
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# 1. Kích hoạt 16GB Linux NVMe Swapfile (Nâng tổng bộ nhớ ảo lên 28.67GB, chống OOM 100%)
!test -f /swapfile || (fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile)

# 2. Phát hiện dung lượng GPU VRAM để tự động cấu hình OLLAMA_MAX_LOADED_MODELS
vram_mb = 16000
gpu_name = "Tesla T4"
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        text=True
    ).strip().splitlines()[0]
    gpu_name, mem_str = [x.strip() for x in out.split(",")]
    vram_mb = int(float(mem_str))
except Exception:
    pass

max_loaded = "2" if vram_mb >= 30000 else "1"

# 3. Cài đặt zstd và Ollama Linux chính thức
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Cấu hình biến môi trường Plan 1: Performance
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"                 # Giữ model trong VRAM / RAM Cache suốt phiên
env["OLLAMA_MAX_LOADED_MODELS"] = max_loaded     # "1" trên T4 16GB (82% VRAM), "2" nếu chạy trên A100 40GB
env["OLLAMA_FLASH_ATTENTION"] = "1"              # Bật Flash Attention tăng tốc prefill & decode
env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"             # Nén KV Cache 8-bit, giảm 50% VRAM cho context 16K

# 5. Khởi chạy 'ollama serve' ngầm
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
clear_output(wait=True)
time.sleep(3)

try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print(f"🎉 Plan 1 Performance Daemon sẵn sàng trên {gpu_name} ({vram_mb/1024:.1f} GB VRAM)!")
        print(f"⚙️ Cấu hình: OLLAMA_MAX_LOADED_MODELS={max_loaded} | FLASH_ATTENTION=1 | KV_CACHE=q8_0 | 16GB Swap")
        print(resp.read().decode("utf-8"))
except Exception as e:
    print("⚠️ Không thể kết nối Ollama daemon:", e)

### Step 4: Tải Bộ Đôi Mô Hình Plan 1 Performance (`deepseek-r1:14b` + `qwen2.5-coder:14b` ⭐) & Warm-up Page Cache
1. **Model 1 — Nhạc Trưởng (`deepseek-r1:14b`):** Chạy 1 lần đầu feature để phân tích yêu cầu `<think>` và xuất checklist `T1..Tn` vào `.agents/kb/HANDOFF.md`.
2. **Model 2 — Chuyên Gia (`qwen2.5-coder:14b` ⭐):** Thực thi xuyên suốt `T1 -> Tn` (Coding, Workflow, Diagram) với 0 lần switch và giữ nguyên KV Cache Prefix.
3. **Warm-up Linux Page Cache:** Nạp trước trọng số vào RAM/Swap cache và ghim sẵn `qwen2.5-coder:14b` trong GPU VRAM.

In [ ]:
import json
import urllib.request
from IPython.display import clear_output

PLAN1_T4_MODELS = [
    ("deepseek-r1:14b",   "🎼 Model 1 (Nhạc Trưởng): DeepSeek CoT <think> Planning & Reasoning (~9.0 GB)"),
    ("qwen2.5-coder:14b", "🛠️ Model 2 (Chuyên Gia ⭐): Coding, Workflow & Mermaid Diagram (~9.5 GB)"),
]

print("⏳ Đang tải bộ đôi mô hình Plan 1: Performance (Tesla T4)...")
for tag, role in PLAN1_T4_MODELS:
    print(f"\n📦 Đang tải {tag} — {role}...")
    !ollama pull {tag}

print("\n🔥 Đang thực hiện Warm-up Linux OS Page Cache & ghim Model 2 (qwen2.5-coder:14b) sẵn sàng...")
for tag, _ in PLAN1_T4_MODELS:
    try:
        payload = json.dumps({"model": tag, "prompt": "hi", "stream": False, "keep_alive": "24h"}).encode("utf-8")
        req = urllib.request.Request("http://localhost:11434/api/generate", data=payload, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=120):
            print(f"  ✨ Đã warm-up cache thành công: {tag}")
    except Exception as e:
        print(f"  ⚠️ Bỏ qua warm-up {tag}: {e}")

clear_output(wait=True)
print("✅ Hoàn tất tải & Warm-up bộ đôi mô hình Plan 1 (Performance T4)!")
!ollama list

### Step 5: Mở Cloudflare Quick Tunnel & Kết Nối Với IDE (Chống Deadlock Pipe 64KB)
Khởi tạo đường hầm HTTPS bảo mật từ `localhost:11434` ra public URL `*.trycloudflare.com`.
> 🛡️ **Scope 3 Hardening:** Tự động kích hoạt luồng nền (`daemon=True`) xả liên tục `tunnel_proc.stdout` sau khi lấy được URL, ngăn chặn triệt để lỗi treo đường hầm (HTTP 524 / Pipe Buffer 64KB Deadlock) khi chạy phiên dài.

In [ ]:
import re
import subprocess
import threading
import time
from IPython.display import clear_output

# 1. Tải và cài đặt Cloudflare Tunnel (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Khởi chạy cloudflared chuyển tiếp HTTPS về localhost:11434
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Đọc stdout để trích xuất URL '*.trycloudflare.com'
tunnel_url = None
print("⏳ Đang thiết lập Cloudflare Quick Tunnel...")
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

# 4. QUAN TRỌNG: Chạy thread nền xả stdout liên tục để tránh đầy bộ đệm OS Pipe 64KB gây treo tunnel
def _drain_tunnel_logs(proc):
    try:
        for _ in iter(proc.stdout.readline, ""):
            pass
    except Exception:
        pass

threading.Thread(target=_drain_tunnel_logs, args=(tunnel_proc,), daemon=True).start()

clear_output(wait=True)

if tunnel_url:
    print("=" * 78)
    print("🎉 PLAN 1: PERFORMANCE (T4 16GB) TUNNEL IS ONLINE!")
    print(f"🔗 PUBLIC ENDPOINT URL: {tunnel_url}")
    print("=" * 78)
    print("\n🚀 CÁCH 1 (Dán trực tiếp URL vào khung chat IDE / CLI):\n")
    print(f"   {tunnel_url}")
    print("   (Agent sẽ tự động gọi `ollama_set_endpoint`, kiểm tra kết nối và chuyển sang GPU Colab!)\n")
    print("⚡ CÁCH 2 (Chạy lệnh 1 dòng trong Terminal tại thư mục dự án):\n")
    print(f"   python3 colab_ollama_mcp/server.py --set-endpoint {tunnel_url}")
    print("\n" + "-" * 78)
else:
    print("❌ Không lấy được Cloudflare Tunnel URL. Vui lòng chạy lại cell này.")

### Step 6: Duy Trì Kết Nối Phiên Dài (HTML5 Silent Audio Keep-Alive)
Phát vòng lặp âm thanh tĩnh 500ms khi mở bằng trình duyệt web (Safari/Chrome), giữ cho tab Colab luôn ở trạng thái *Media Active* và không bị đóng băng hay tự động tải lại (`F5`).

In [ ]:
from IPython.display import HTML, display

display(HTML('''
<div style="padding: 14px; background: #e8f5e9; border: 1px solid #4caf50; border-radius: 8px; font-family: sans-serif;">
  <b style="color: #2e7d32;">🟢 Keep-Alive Active (Silent Audio Loop)</b>
  <p style="margin: 4px 0 8px 0; color: #1b5e20; font-size: 13px;">
     Giữ phiên Google Colab luôn hoạt động ổn định trong suốt quá trình bạn code trên IDE.
  </p>
  <audio loop autoplay controls style="height: 30px;">
    <source src="data:audio/wav;base64,UklGRsQAAABXQVZFZm10IBAAAAABAAEAQB8AAIA+AAACABAAZGF0YaAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA" type="audio/wav">
  </audio>
</div>
'''))

### Step 7: Bảng Theo Dõi Tài Nguyên Thời Gian Thực (GPU VRAM, System RAM, Swap & Models Trong VRAM)
Chạy cell này bất kỳ lúc nào để kiểm tra **GPU VRAM**, **System RAM + Swapfile**, **OS Page Cache**, **SSD Disk**, và **Danh sách Model đang ghim thực tế trong GPU VRAM (`/api/ps`)**.
- Để `AUTO_REFRESH = False` (mặc định) để xem kết quả tức thì.
- Đổi `AUTO_REFRESH = True` để bật bảng theo dõi tự động làm mới mỗi 5 giây (tốn **0 MB** bộ nhớ DOM nhờ `clear_output(wait=True)`).

In [ ]:
import json
import shutil
import subprocess
import time
import urllib.request
from IPython.display import clear_output

AUTO_REFRESH = False
REFRESH_SECONDS = 5

def make_bar(pct: float, width: int = 24) -> str:
    filled = max(0, min(width, int(round((pct / 100.0) * width))))
    return "█" * filled + "░" * (width - filled)

def print_telemetry() -> None:
    print("=" * 78)
    print(f"📊 PLAN 1 PERFORMANCE (T4) TELEMETRY — {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
    print("=" * 78)

    # 1. GPU VRAM, Load & Temperature
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True).strip()
        for line in out.splitlines():
            name, temp, util, mem_used, mem_total = [x.strip() for x in line.split(",")]
            used_gb = float(mem_used) / 1024.0
            total_gb = float(mem_total) / 1024.0
            vram_pct = (float(mem_used) / float(mem_total)) * 100.0 if float(mem_total) > 0 else 0.0
            util_pct = float(util)
            print(f"🎮 GPU Hardware : {name}  (🌡️ {temp}°C)")
            print(f"⚡ GPU Core Load: [{make_bar(util_pct)}] {util_pct:5.1f}%")
            print(f"🧠 GPU VRAM     : [{make_bar(vram_pct)}] {vram_pct:5.1f}%  ({used_gb:.2f} GB / {total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ GPU Status   : {e}")

    # 2. System RAM, Page Cache & Swapfile
    try:
        meminfo = {}
        with open("/proc/meminfo", "r") as f:
            for line in f:
                parts = line.split(":")
                if len(parts) == 2:
                    meminfo[parts[0].strip()] = int(parts[1].strip().split()[0])
        total_ram_gb = meminfo.get("MemTotal", 0) / (1024 ** 2)
        avail_ram_gb = meminfo.get("MemAvailable", 0) / (1024 ** 2)
        used_ram_gb = total_ram_gb - avail_ram_gb
        ram_pct = (used_ram_gb / total_ram_gb * 100.0) if total_ram_gb > 0 else 0.0
        cached_gb = meminfo.get("Cached", 0) / (1024 ** 2)
        swap_total_gb = meminfo.get("SwapTotal", 0) / (1024 ** 2)
        swap_free_gb = meminfo.get("SwapFree", 0) / (1024 ** 2)
        swap_used_gb = swap_total_gb - swap_free_gb
        print(f"💻 System RAM   : [{make_bar(ram_pct)}] {ram_pct:5.1f}%  ({used_ram_gb:.2f} GB / {total_ram_gb:.2f} GB | Page Cache: {cached_gb:.2f} GB)")
        print(f"🛡️ NVMe Swapfile: {swap_used_gb:.2f} GB / {swap_total_gb:.2f} GB (Tổng bộ nhớ ảo: {total_ram_gb + swap_total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ System RAM   : {e}")

    # 3. Colab SSD Disk Usage
    try:
        total_d, used_d, _ = shutil.disk_usage("/")
        used_d_gb = used_d / (1024 ** 3)
        total_d_gb = total_d / (1024 ** 3)
        disk_pct = (used_d / total_d * 100.0) if total_d > 0 else 0.0
        print(f"💾 Colab Disk   : [{make_bar(disk_pct)}] {disk_pct:5.1f}%  ({used_d_gb:.1f} GB / {total_d_gb:.1f} GB)")
    except Exception as e:
        print(f"⚠️ Colab Disk   : {e}")

    # 4. Active Models Pinned in GPU VRAM (/api/ps)
    print("-" * 78)
    try:
        with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
            ps_data = json.loads(resp.read().decode("utf-8"))
            running = ps_data.get("models", [])
            if running:
                print(f"🔥 Đang ghim trong GPU VRAM ({len(running)} model):")
                for m in running:
                    m_name = m.get("name", "unknown")
                    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
                    total_gb = m.get("size", 0) / (1024 ** 3)
                    gpu_offload = (vram_gb / total_gb * 100.0) if total_gb > 0 else 100.0
                    print(f"   • {m_name} — VRAM: {vram_gb:.2f} GB ({gpu_offload:.0f}% GPU offload)")
            else:
                print("💤 GPU VRAM hiện đang rảnh (Model sẽ nạp tức thì khi có request đầu tiên)")
    except Exception as e:
        print(f"🔴 Ollama Status: Offline hoặc không phản hồi ({e})")
    print("=" * 78)

try:
    while True:
        clear_output(wait=True)
        print_telemetry()
        if not AUTO_REFRESH:
            break
        print(f"🔄 Tự động làm mới mỗi {REFRESH_SECONDS}s... (Nhấn Stop ⏹️ để dừng)")
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("⏹️ Đã dừng bảng theo dõi tài nguyên.")